# Benchmark: pandas + openpyxl vs Polars + calamine

Notebook para comparar, de forma simples, duas abordagens de leitura de arquivos Excel (`.xlsx`).

- `pandas.read_excel(..., engine="openpyxl")`
- `polars.read_excel(..., engine="calamine")`


## Dependencias

Instale as dependencias antes de executar o notebook:

```bash
pip install pandas openpyxl polars fastexcel tqdm
```


In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import polars as pl
from tqdm import tqdm


In [ ]:
PASTA_BASES = Path(r"C:\\Python\\Projeto_Conciliacao_Contabil\\_Bases_Razoes")
ABA = "Exportação SAPUI5"
COLUNAS = ["Conta do Razão", "Mont.moeda empresa"]

arquivos = sorted(PASTA_BASES.glob("*.xlsx"))
print(f"Arquivos encontrados: {len(arquivos)}")


## Leitura com pandas + openpyxl


In [ ]:
inicio = perf_counter()
dataframes_pandas = []

for arquivo in tqdm(arquivos, desc="pandas/openpyxl", unit="arq"):
    df = pd.read_excel(
        arquivo,
        sheet_name=ABA,
        usecols=COLUNAS,
        engine="openpyxl",
    )
    df["arquivo_origem"] = arquivo.name
    dataframes_pandas.append(df)

if dataframes_pandas:
    df_pandas = pd.concat(dataframes_pandas, ignore_index=True)
else:
    df_pandas = pd.DataFrame(columns=COLUNAS + ["arquivo_origem"])

tempo_pandas = perf_counter() - inicio

print(f"Linhas carregadas: {len(df_pandas)}")
print(f"Tempo pandas/openpyxl: {tempo_pandas:.2f} segundos")
df_pandas.head()


## Leitura com Polars + calamine


In [ ]:
inicio = perf_counter()
dataframes_polars = []

for arquivo in tqdm(arquivos, desc="polars/calamine", unit="arq"):
    df = pl.read_excel(
        arquivo,
        sheet_name=ABA,
        columns=COLUNAS,
        engine="calamine",
    )
    df = df.with_columns(pl.lit(arquivo.name).alias("arquivo_origem"))
    dataframes_polars.append(df)

if dataframes_polars:
    df_polars = pl.concat(dataframes_polars)
else:
    df_polars = pl.DataFrame({coluna: [] for coluna in COLUNAS + ["arquivo_origem"]})

tempo_polars = perf_counter() - inicio

print(f"Linhas carregadas: {df_polars.height}")
print(f"Tempo Polars/calamine: {tempo_polars:.2f} segundos")
df_polars.head()


## Comparacao final


In [ ]:
resultado = pd.DataFrame(
    [
        {
            "metodo": "pandas + openpyxl",
            "arquivos": len(arquivos),
            "linhas": len(df_pandas),
            "tempo_segundos": round(tempo_pandas, 2),
        },
        {
            "metodo": "Polars + calamine",
            "arquivos": len(arquivos),
            "linhas": df_polars.height,
            "tempo_segundos": round(tempo_polars, 2),
        },
    ]
)

resultado


In [ ]:
if tempo_polars > 0:
    ganho = tempo_pandas / tempo_polars
    print(f"Polars/calamine foi {ganho:.2f}x mais rapido que pandas/openpyxl.")
else:
    print("Tempo do Polars ficou abaixo da precisao medida.")
